# Data Simulator

Generates synthetic training movies for `EventDetector`: binding, unbinding,
and movement (dipole) events composited onto real instrument background/noise
sampled from recorded buffer movies.

In [2]:
import sys
from pathlib import Path

import numpy as np
import torch
from abc import ABC, abstractmethod
from dataclasses import dataclass
from functools import partial
from typing import Sequence

from alex_area.movie_generator.buffer_movies import BufferMovie, load_buffer_movies
from alex_area.movie_generator.sim_movie import CachedPSF, BufferMovieSim
from alex_area import utils

project_root = Path(r"c:\Users\chem-bras5436\Documents\ml_mp_movements")
sys.path.insert(0, str(project_root))

from model.model import EventDetector

In [3]:
MIN_FRAME_DIM_SIZE: int = 42  # smallest crop width/height sampled by gen_random_mov_stack
BORDER_MASK: int = 5  # px excluded from each edge when placing events, keeping the full PSF footprint in-frame
NM_PER_PX: float = 72.6  # spatial calibration (nm/pixel)
FRAMES_PER_SECOND: float = 42.7  # acquisition frame rate (Hz)
HEATMAP_GAUSSIAN_SIGMA_PX: float = 1.0  # stdev of the ground truth heatmap gaussian, in px
HEATMAP_GAUSSIAN_THUMBNAIL_SIZE: int = 9  # px window the gaussian is truncated to (must fit within BORDER_MASK)

In [4]:
B_MOV_X_MAX: int = 231
B_MOV_Y_MAX: int = 163

# Buffer movies capture real instrument background/noise with no particle events;
# indices 12+ correspond to TwoMP large-FoV buffer movies.
b_movs = load_buffer_movies()[12:]
BUFFER_MOVIES = [
    mov[:, :B_MOV_Y_MAX, :B_MOV_X_MAX] for mov in b_movs
]  # crop all buffer movies to a common (Y, X) footprint

In [5]:
def density_to_n_events(
    density: float,
    shape: tuple[int, int, int],
) -> int:
    """Convert an event density to an expected event count for a movie.

    Args:
        density: Event density, in events / um^2 / s.
        shape: Movie array shape (T, H, W).

    Returns:
        Expected number of events over the movie's full duration and area.
    """
    area_um2 = (shape[1] * NM_PER_PX / 1000) * (shape[2] * NM_PER_PX / 1000)
    duration_s = shape[0] / FRAMES_PER_SECOND

    return int(density * area_um2 * duration_s)

In [6]:
def gen_random_mov_stack(length: int = 500, mov_thumbnail_size: int = 64) -> BufferMovie:
    """Sample a random spatiotemporal crop from a randomly chosen buffer movie.

    Selects one of `BUFFER_MOVIES` at random, then crops it to a
    `mov_thumbnail_size` x `mov_thumbnail_size` (y, x) window and a
    `length`-frame time window, each placed at a random offset. Used to vary
    the background content and temporal window seen during training.

    Args:
        length: Number of frames to crop from the movie's time axis.
        mov_thumbnail_size: Width and height, in px, of the cropped spatial
            window.

    Returns:
        The cropped buffer movie, shape (length, mov_thumbnail_size,
        mov_thumbnail_size).
    """
    rand_index = np.random.randint(0, len(BUFFER_MOVIES))
    mov: BufferMovie = BUFFER_MOVIES[rand_index]

    x_width = mov_thumbnail_size
    y_width = mov_thumbnail_size

    x_max = B_MOV_X_MAX - x_width
    y_max = B_MOV_Y_MAX - y_width
    t_max = mov.shape[0] - length

    y_rand = np.random.randint(0, y_max + 1)
    x_rand = np.random.randint(0, x_max + 1)
    t_rand = np.random.randint(0, t_max + 1)

    return mov[
        t_rand : t_rand + length, y_rand : y_rand + y_width, x_rand : x_rand + x_width
    ]

In [7]:
class AbstractSimEvent(ABC):
    """Interface for a single simulated event placed into a training movie.

    Concrete events store their (x, y, i, c) coordinates as plain fields;
    this interface only constrains the values derived from them.
    """

    x: float
    y: float
    i: float
    c: float

    @property
    @abstractmethod
    def hot_px(self) -> tuple[int, int]:
        """Nearest integer (x, y) pixel, i.e. the heatmap peak location."""

    @property
    @abstractmethod
    def offset(self) -> tuple[float, float]:
        """Sub-pixel (dx, dy) offset of (x, y) from `hot_px`."""

    @abstractmethod
    def to_simple(self) -> list[list[float]]:
        """Flatten the event to one or more [x, y, i, c] records."""

In [8]:
@dataclass
class BaseSimEvent(AbstractSimEvent):
    """Concrete single-point event: a binding or unbinding at (x, y, i, c).

    Attributes:
        x: Sub-pixel x-coordinate.
        y: Sub-pixel y-coordinate.
        i: Frame index (temporal coordinate).
        c: Contrast.
    """

    x: float
    y: float
    i: float
    c: float

    @property
    def hot_px(self) -> tuple[int, int]:
        return (np.round(self.x).astype(int), np.round(self.y).astype(int))

    @property
    def offset(self) -> tuple[float, float]:
        x_px, y_px = self.hot_px

        return (self.x - float(x_px), self.y - float(y_px))

    def to_simple(self) -> list[list[float]]:
        return [[self.x, self.y, self.i, self.c]]

In [9]:
class BindingSimEvent(BaseSimEvent):
    """A particle landing (binding) event."""


class UnbindingSimEvent(BaseSimEvent):
    """A particle leaving (unbinding) event."""


@dataclass
class MovementSimEvent(BaseSimEvent):
    """A movement: a linked unbinding-then-binding pair.

    Represents a particle moving from one location to another, modeled as an
    unbinding event and a binding event straddling the midpoint (x, y),
    separated by `distance` at angle `theta`. Both endpoints share the same
    intensity/frame index and contrast.

    Attributes:
        distance: Distance between the unbinding and binding endpoints.
        theta: Direction of travel, in radians.
    """

    distance: float
    theta: float

    def __post_init__(self) -> None:
        self.theta = (self.theta / (2 * np.pi)) - np.floor(
            self.theta / (2 * np.pi)
        )  # wrap to [0, 2*pi)

        self.dx: float = self.distance * np.cos(self.theta)
        self.dy: float = self.distance * np.sin(self.theta)

        self.unbinding: UnbindingSimEvent = UnbindingSimEvent(
            x=self.x - self.dx / 2, y=self.y - self.dy / 2, i=self.i, c=-self.c
        )
        self.binding: BindingSimEvent = BindingSimEvent(
            x=self.x + self.dx / 2, y=self.y + self.dy / 2, i=self.i, c=self.c
        )

    def to_simple(self) -> list[list[float]]:
        """Flatten to the unbinding and binding endpoint records.

        Returns:
            A list of two [x, y, i, c] records: the unbinding endpoint
            followed by the binding endpoint.
        """
        return self.unbinding.to_simple() + self.binding.to_simple()

In [10]:
def gen_events(
    event_density: float,
    mov_shape: tuple[int, int, int],
    contrast_range: tuple[float, float],
    distance_range: tuple[float, float] = (1.5, 300),
    event_type_weight: tuple[float, float, float] = (1, 1, 1),
    navg: int = 5,
) -> Sequence[AbstractSimEvent]:
    """Sample a batch of random binding, unbinding, and movement events.

    Args:
        event_density: Target combined event density, in events / um^2 / s,
            split across the three event types by `event_type_weight`.
        mov_shape: Movie array shape (T, H, W) events are placed within.
        contrast_range: (low, high) contrast sampled for each event's
            arrival endpoint; departures use the negated contrast.
        distance_range: (low, high) pixel distance sampled for each
            movement event's unbinding-binding separation.
        event_type_weight: Relative weighting of (binding, unbinding,
            movement) events; need not sum to 1.
        navg: Number of frames averaged into each side of a ratiometric
            window (see `gen_ratiometric_movie`). Events are never placed in
            the first/last `navg` frames, since those have no valid
            ratiometric value -- avoiding any need to handle them downstream.

    Returns:
        The generated events, in no particular order.

    Raises:
        ValueError: If `event_type_weight` doesn't have exactly 3 elements.
    """
    if len(event_type_weight) != 3:
        raise ValueError("length of weights must be equal to no. of event types")

    weights = np.array(event_type_weight) / np.sum(event_type_weight)

    mov_shape_masked = (
        mov_shape[0],
        mov_shape[1] - 2 * BORDER_MASK,
        mov_shape[2] - 2 * BORDER_MASK,
    )

    n_bindings, n_unbindings, n_movements = (
        density_to_n_events(density=density, shape=mov_shape_masked)
        for density in weights * event_density
    )

    # half-pixel margin inside BORDER_MASK so a rounded hot_px never falls in the masked border
    x_low, x_high = BORDER_MASK - 0.5, mov_shape[2] - (BORDER_MASK - 0.5)
    y_low, y_high = BORDER_MASK - 0.5, mov_shape[1] - (BORDER_MASK - 0.5)

    # exclude the first/last navg frames, which have no valid ratiometric value
    i_low, i_high = navg, mov_shape[0] - navg

    binding_evs = [
        BindingSimEvent(x=x, y=y, i=i, c=c)
        for x, y, i, c in zip(
            np.random.uniform(x_low, x_high, n_bindings),
            np.random.uniform(y_low, y_high, n_bindings),
            np.random.uniform(i_low, i_high, n_bindings),
            np.random.uniform(contrast_range[0], contrast_range[1], n_bindings),
        )
    ]
    unbinding_evs = [
        UnbindingSimEvent(x=x, y=y, i=i, c=c)
        for x, y, i, c in zip(
            np.random.uniform(x_low, x_high, n_unbindings),
            np.random.uniform(y_low, y_high, n_unbindings),
            np.random.uniform(i_low, i_high, n_unbindings),
            -np.random.uniform(contrast_range[0], contrast_range[1], n_unbindings),
        )
    ]

    n_small_movements = int(0.6 * n_movements)
    n_medium_movements = int(0.3 * n_movements)
    n_large_movements = n_movements - n_small_movements - n_medium_movements

    movement_evs = [
        MovementSimEvent(x=x, y=y, i=i, c=c, distance=distance / NM_PER_PX, theta=theta)
        for x, y, i, c, distance, theta in zip(
            np.random.uniform(x_low, x_high, n_movements),
            np.random.uniform(y_low, y_high, n_movements),
            np.random.uniform(i_low, i_high, n_movements),
            np.random.uniform(contrast_range[0], contrast_range[1], n_movements),
            np.concatenate(
                [
                    np.random.uniform(distance_range[0], 20.0, n_small_movements),
                    np.random.uniform(20.0, 100.0, n_medium_movements),
                    np.random.uniform(100.0, distance_range[1], n_large_movements),
                ]
            ),
            np.random.uniform(0, 2 * np.pi, n_movements),
        )
    ]

    return binding_evs + unbinding_evs + movement_evs

In [11]:
def gen_ground_truth(
    events: Sequence[AbstractSimEvent],
    mov_shape: tuple[int, int, int],
    sigma_px: float = HEATMAP_GAUSSIAN_SIGMA_PX,
) -> dict[str, torch.Tensor]:
    """Render simulated events into ground truth compatible with `loss_fn`.

    For each event, adds a small Gaussian thumbnail (peak 1, truncated to
    `HEATMAP_GAUSSIAN_THUMBNAIL_SIZE` px) onto its class's heatmap centered
    at (frame, hot_px) -- the CornerNet/CenterNet-style target
    `loss_fn_heatmap` expects -- and records its sub-pixel (dy, dx) offset at
    that same peak voxel. Movement events additionally record their (cos,
    sin) orientation there. Overlapping events of the same class are summed,
    then the heatmap is clipped to [0, 1] so nearby peaks still saturate to
    1 rather than exceeding it.

    Args:
        events: Simulated events, as returned by `gen_events`.
        mov_shape: Movie array shape (T, H, W) the events were placed within.
        sigma_px: Standard deviation, in px, of the heatmap Gaussian.

    Returns:
        Dict with keys "heatmap" (3, T, H, W), "offset" (2, T, H, W), and
        "orientation" (2, T, H, W) -- matching `predictions` from
        `EventDetector.forward` up to the batch dimension.

    Raises:
        TypeError: If `events` contains a type other than `BindingSimEvent`,
            `UnbindingSimEvent`, or `MovementSimEvent`.
    """
    t_size, h_size, w_size = mov_shape

    heatmap = np.zeros((3, t_size, h_size, w_size), dtype=np.float32)
    offset = np.zeros((2, t_size, h_size, w_size), dtype=np.float32)
    orientation = np.zeros((2, t_size, h_size, w_size), dtype=np.float32)

    half = HEATMAP_GAUSSIAN_THUMBNAIL_SIZE // 2
    patch_yy, patch_xx = np.mgrid[-half : half + 1, -half : half + 1]
    gaussian_patch = np.exp(-(patch_xx**2 + patch_yy**2) / (2 * sigma_px**2))

    for event in events:
        if isinstance(event, BindingSimEvent):
            channel = EventDetector.CLASS_BINDING
        elif isinstance(event, UnbindingSimEvent):
            channel = EventDetector.CLASS_UNBINDING
        elif isinstance(event, MovementSimEvent):
            channel = EventDetector.CLASS_MOVEMENT
        else:
            raise TypeError(f"unrecognized event type: {type(event).__name__}")

        frame = int(np.clip(np.round(event.i), 0, t_size - 1))
        x_px, y_px = event.hot_px
        dx, dy = event.offset

        heatmap[
            channel,
            frame,
            y_px - half : y_px + half + 1,
            x_px - half : x_px + half + 1,
        ] += gaussian_patch

        # last write wins: if another event already claimed this (frame, hot_px)
        # voxel -- same class or not -- its offset/orientation is silently
        # overwritten rather than averaged. Unlike the heatmap, a single voxel
        # can't represent two distinct sub-pixel positions/angles, and with
        # continuous-valued placement over a large voxel grid this is rare
        # enough not to be worth tracking/averaging.
        offset[:, frame, y_px, x_px] = (dy, dx)

        if isinstance(event, MovementSimEvent):
            orientation[:, frame, y_px, x_px] = (np.cos(event.theta), np.sin(event.theta))

    heatmap = np.clip(heatmap, 0.0, 1.0)

    return {
        "heatmap": torch.from_numpy(heatmap),
        "offset": torch.from_numpy(offset),
        "orientation": torch.from_numpy(orientation),
    }

In [ ]:
def gen_ratiometric_movie(movie: np.ndarray, navg: int) -> np.ndarray:
    """Convert a raw movie to ratiometric contrast frames.

    At each valid center frame, sums the `navg` frames immediately before it
    (trailing) and the `navg` frames immediately after it (running) --
    excluding the center frame itself from both -- then computes
    `running / trailing - 1`. Frames too close to either end of the movie to
    have a full `navg`-frame window on both sides are set to NaN, exactly
    `navg` frames at each end -- matching the exclusion zone `gen_events`'
    `navg` argument keeps events out of.

    Args:
        movie: Raw movie, shape (T, H, W).
        navg: Number of frames summed into each trailing/running window.

    Returns:
        Ratiometric movie, same shape as `movie`, dtype float32. The first
        and last `navg` frames are NaN.

    Raises:
        ValueError: If `movie` has fewer than `2 * navg + 1` frames.
    """
    t_size = movie.shape[0]
    if t_size < 2 * navg + 1:
        raise ValueError("movie is too short for the given navg")

    centers = np.arange(navg, t_size - navg)

    # padded_cumsum[k] == movie[:k].sum(axis=0), so any window's sum is a
    # single subtraction instead of re-summing overlapping frames per center
    padded_cumsum = np.concatenate(
        [
            np.zeros((1, *movie.shape[1:]), dtype=np.float64),
            np.cumsum(movie, axis=0, dtype=np.float64),
        ]
    )

    trailing = padded_cumsum[centers] - padded_cumsum[centers - navg]
    running = padded_cumsum[centers + navg + 1] - padded_cumsum[centers + 1]

    eps = np.finfo(np.float64).eps
    ratio = running / (trailing + eps) - 1.0

    ratiometric = np.full(movie.shape, np.nan, dtype=np.float32)
    ratiometric[centers] = ratio.astype(np.float32)

    return ratiometric

In [12]:
PSF_PATH = (
    r"C:\Users\chem-bras5436\Documents\MP_DATA\citrate_synthase\TwoMP"
    r"\buffer_movies\unbinned\011_20241115_s_elo_wt_his_tag_7500x_expPSF.pickle"
)
_raw_psf = utils.load_from_pickle(PSF_PATH)

def expPSF(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Evaluate the experimental PSF at arbitrary (x, y) coordinates.

    Thin wrapper around the raw RectBivariateSpline interpolator that fixes
    ``grid=False`` so callers can pass flat coordinate arrays directly.

    Parameters
    ----------
    x, y :
        Coordinate arrays in pixel units, as returned by
        ``CachedPSF._gen_offset_mgrid``.

    Returns
    -------
    np.ndarray
        PSF amplitude at each coordinate pair.
    """
    return _raw_psf(x, y, grid=False)

ePSF = CachedPSF(expPSF, l_thum=21)

simulator = BufferMovieSim(psf_model=ePSF)

In [13]:
calib = BUFFER_MOVIES[0].calibration
c2m = lambda c: np.divide(np.subtract(c, calib["intercept"]), calib["c2kDa"])
m2c = lambda m: np.add(np.multiply(m, calib["c2kDa"]), calib["intercept"])

In [14]:
def gen_doped_events(
    event_density: float,
    mov_shape: tuple[int, int, int],
    contrast_range: tuple[float, float],
    dopant_contrast_range: tuple[float, float],
    distance_range: tuple[float, float] = (1.5, 300),
    event_type_weight: tuple[float, float, float] = (1, 1, 1),
    dopant_density_fraction: float = 0.2,
    navg: int = 5,
) -> Sequence[AbstractSimEvent]:
    """Wraps `gen_events`, adding binding/unbinding-only "dopant" events at
    `dopant_contrast_range` so the model also sees events outside the
    primary population's mass range.

    Args:
        dopant_contrast_range: (low, high) contrast for the dopant events.
        dopant_density_fraction: Dopant density as a fraction of
            `event_density`, split evenly between binding and unbinding.
        See `gen_events` for the remaining args.

    Returns:
        The primary and dopant events combined, in no particular order.
    """
    events = gen_events(
        event_density=event_density,
        mov_shape=mov_shape,
        contrast_range=contrast_range,
        distance_range=distance_range,
        event_type_weight=event_type_weight,
        navg=navg,
    )
    dopant_events = gen_events(
        event_density=dopant_density_fraction * event_density,
        mov_shape=mov_shape,
        contrast_range=dopant_contrast_range,
        distance_range=distance_range,
        event_type_weight=(1, 1, 0),
        navg=navg,
    )

    return events + dopant_events

In [15]:
event_generator = partial(
    gen_doped_events,
    contrast_range=(m2c(2000), m2c(6000)),
    dopant_contrast_range=(m2c(30), m2c(2000)),
)

In [ ]:
OPTIMUM_EVENT_DENSITY = 0.5
N = 100_000  # not sure what to name this variable
event_densities = np.random.uniform(
    OPTIMUM_EVENT_DENSITY, 5 * OPTIMUM_EVENT_DENSITY, size=N
)

for density in event_densities:
    rand_mov_stack = gen_random_mov_stack()

    generated_events = event_generator(
        event_density=density, mov_shape=rand_mov_stack.shape
    )

    simple_events = []

    for event in generated_events:
        simple_events += event.to_simple()

    ground_truth = gen_ground_truth(
        events=generated_events, mov_shape=rand_mov_stack.shape
    )

    sim_movie = simulator.work(movie=rand_mov_stack, events=simple_events)

In [17]:
# start timer

import time


start_time = time.time()

for i in range(64):    
    rand_mov_stack = gen_random_mov_stack()

    generated_events = event_generator(
        event_density=2.5, mov_shape=rand_mov_stack.shape
    )

    simple_events = []

    for event in generated_events:
        simple_events += event.to_simple()

    ground_truth = gen_ground_truth(
        events=generated_events, mov_shape=rand_mov_stack.shape
    )

    sim_movie = simulator.work(movie=rand_mov_stack, events=simple_events)

    print(f"Iteration {i+1}/64 complete. Elapsed time: {time.time() - start_time:.2f} seconds.")


Adding events: 100%|██████████| 684/684 [00:00<00:00, 1365.54events/s]


Iteration 1/64 complete. Elapsed time: 0.59 seconds.


Adding events: 100%|██████████| 684/684 [00:00<00:00, 1305.89events/s]


Iteration 2/64 complete. Elapsed time: 1.17 seconds.


Adding events: 100%|██████████| 684/684 [00:00<00:00, 1344.65events/s]


...

Iteration 62/64 complete. Elapsed time: 34.35 seconds.


Adding events: 100%|██████████| 684/684 [00:00<00:00, 1488.93events/s]


Iteration 63/64 complete. Elapsed time: 34.86 seconds.


Adding events: 100%|██████████| 684/684 [00:00<00:00, 1383.54events/s]

Iteration 64/64 complete. Elapsed time: 35.41 seconds.
